In [1]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd

# ---------- User-configurable paths ----------
path_data_intermediate = "/Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"  # <-- change this
os.makedirs(path_data_intermediate, exist_ok=True)

parquet_out = os.path.join(path_data_intermediate, "CompustatPensions.parquet")
csv_out     = os.path.join(path_data_intermediate, "CompustatPensions.csv")

In [2]:
#1 Load Compustat Data

SQL = """
SELECT
    a.gvkey,
    a.datadate,
    a.paddml,
    a.pbnaa,
    a.pbnvv,
    a.pbpro,
    a.pbpru,
    a.pcupsu,
    a.pplao,
    a.pplau
FROM comp.aco_pnfnda AS a
WHERE a.consol = 'C'
  AND a.popsrc = 'D'
  AND a.datafmt = 'STD'
  AND a.indfmt = 'INDL'
  AND a.datadate >= DATE '2000-01-01';
"""


In [3]:
#2 Compustat Data Extraction From WRDS

db = wrds.Connection()
df = db.raw_sql(SQL, date_cols=["datadate"])

Enter your WRDS username [nglei]: nglei2025
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [4]:
print(df)

        gvkey   datadate  paddml  pbnaa  pbnvv    pbpro  pbpru  pcupsu  \
0      001183 2000-01-31    <NA>   <NA>   <NA>     <NA>   <NA>    <NA>   
1      001240 2000-01-31     0.0   <NA>   <NA>    423.0   <NA>    <NA>   
2      001594 2000-01-31    <NA>   <NA>   <NA>     <NA>   <NA>    <NA>   
3      001655 2000-01-31    <NA>   <NA>   <NA>   17.938   <NA>    <NA>   
4      001864 2000-01-31    <NA>   <NA>   <NA>     <NA>   <NA>    <NA>   
...       ...        ...     ...    ...    ...      ...    ...     ...   
65298  210216 2025-06-30    <NA>   <NA>   <NA>  799.886   <NA>    <NA>   
65299  002663 2025-07-31    <NA>   <NA>   <NA>   1214.0   <NA>    <NA>   
65300  004036 2025-07-31    <NA>   <NA>   <NA>    402.4   <NA>    <NA>   
65301  027928 2025-07-31    <NA>   <NA>   <NA>     <NA>   <NA>    <NA>   
65302  165798 2025-07-31    <NA>   <NA>   <NA>      0.0   <NA>    <NA>   

         pplao  pplau  
0         <NA>   <NA>  
1        582.0   <NA>  
2         <NA>   <NA>  
3       21.276 

In [5]:
#3 Data Cleaning

# ---------------- year = year(datadate) + 1 ----------------
df["year"] = df["datadate"].dt.year + 1

# If you want the *effective* analysis year to be 2000+ (after the +1 lag), uncomment:
# df = df[df["year"] >= 2000].copy()

# ---------------- Keep earliest datadate within (gvkey, year) ----------------
df = df.sort_values(["gvkey", "year", "datadate"], kind="mergesort")
df = df.groupby(["gvkey", "year"], as_index=False).head(1)

# ---------------- Drop datadate ----------------
df = df.drop(columns=["datadate"])

# ---------------- Optional: destring gvkey ----------------
df["gvkey"] = pd.to_numeric(df["gvkey"], errors="ignore")

# ---------------- mdesc-style missingness summary ----------------
def mdesc_like(frame: pd.DataFrame) -> pd.DataFrame:
    n = len(frame)
    out = pd.DataFrame({
        "non_missing": frame.notna().sum(),
        "missing": frame.isna().sum(),
    })
    out["missing_pct"] = (out["missing"] / n * 100).round(2)
    return out

print("Missingness summary (mdesc-like):")
print(mdesc_like(df))

# ---------------- Save ----------------
df.to_parquet(parquet_out, index=False)
df.to_csv(csv_out, index=False)

print("Saved:")
print(" -", parquet_out)
print(" -", csv_out)

Missingness summary (mdesc-like):
        non_missing  missing  missing_pct
gvkey         65232        0         0.00
paddml         4774    60458        92.68
pbnaa             2    65230       100.00
pbnvv             2    65230       100.00
pbpro         55274     9958        15.27
pbpru            11    65221        99.98
pcupsu            9    65223        99.99
pplao         54753    10479        16.06
pplau            11    65221        99.98
year          65232        0         0.00
Saved:
 - /Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate/CompustatPensions.parquet
 - /Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate/CompustatPensions.csv


/var/folders/r1/svmq3dcj6q761dvhg47857_c0000gn/T/ipykernel_18442/3445682093.py:17: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df["gvkey"] = pd.to_numeric(df["gvkey"], errors="ignore")
